In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
kaggle_df = pd.read_csv('../../datascience/data/raw/weather_national/national_weather_history.csv')

In [3]:
with open("../../datascience/data/raw/cpcb/cpcb_backfill_progress.json") as f:
    live_data = json.load(f)
live_df = pd.DataFrame(live_data)

In [4]:
live_pivoted = live_df.pivot_table(
    index=["sensor_id", "lat", "lon", "date"], columns="parameter", values="value"
).reset_index()

In [5]:
print(kaggle_df.shape, kaggle_df.dtypes, sep="\n")
print(kaggle_df.head(3).to_string())

print(live_df.shape, live_df.dtypes, sep="\n")
print(live_df.head(5).to_string())
print(live_df["parameter"].value_counts())   # which parameter names exist (pm25 vs PM2.5, etc.)

print(live_pivoted.shape, live_pivoted.columns.tolist())
print(live_pivoted.head(3).to_string())

(28509, 7)
city              object
lat              float64
lon              float64
date              object
temp_max         float64
temp_min         float64
precipitation    float64
dtype: object
     city       lat       lon        date  temp_max  temp_min  precipitation
0  Cumbum  15.58171  79.11059  2023-01-01      31.5      19.2            0.0
1  Cumbum  15.58171  79.11059  2023-01-02      30.4      19.0            0.0
2  Cumbum  15.58171  79.11059  2023-01-03      30.3      18.7            0.0
(113341, 7)
sensor_id          int64
lat              float64
lon              float64
parameter         object
location_name     object
date              object
value            float64
dtype: object
   sensor_id        lat        lon parameter            location_name                       date  value
0   12234782  28.563262  77.186937        co  R K Puram, Delhi - DPCC  2026-05-16T00:00:00+05:30  0.906
1   12234782  28.563262  77.186937        co  R K Puram, Delhi - DPCC  2026-05-17T0

In [6]:
import pandas as pd
import json

kaggle_df = pd.read_csv('../../datascience/data/raw/weather_national/national_weather_history.csv')

with open("../../datascience/data/raw/cpcb/cpcb_backfill_progress.json") as f:
    live_data = json.load(f)
live_df = pd.DataFrame(live_data)

live_df["date"] = pd.to_datetime(live_df["date"].str[:10])

# one lat/lon per station
loc = live_df.groupby("location_name")[["lat", "lon"]].first().reset_index()

live_pivoted = live_df.pivot_table(
    index=["location_name", "date"], columns="parameter",
    values="value", aggfunc="mean"
).reset_index()
live_pivoted.columns.name = None

live_pivoted = live_pivoted.merge(loc, on="location_name")
live_pivoted = live_pivoted.rename(columns={
    "pm25": "PM2.5", "pm10": "PM10", "no2": "NO2",
    "so2": "SO2", "co": "CO", "o3": "O3"})
live_pivoted["month"] = live_pivoted["date"].dt.month
live_pivoted["day_of_week"] = live_pivoted["date"].dt.dayofweek

print(live_pivoted.shape)
print(live_pivoted.isna().mean()) 

common_cols = ["lat", "lon", "PM2.5", "PM10", "NO2", "month", "day_of_week"]
for col in common_cols:
    if col not in kaggle_df.columns:
        print(f"WARNING: {col} missing from kaggle_df")
    if col not in live_pivoted.columns:
        print(f"WARNING: {col} missing from live_pivoted")

combined_aqi = pd.concat([kaggle_df, live_pivoted], ignore_index=True)
combined_aqi = combined_aqi.dropna(subset=["lat", "lon"])
print(f"Combined AQI training rows: {len(combined_aqi)}")
print(f"Unique locations: {combined_aqi[['lat','lon']].drop_duplicates().shape[0]}")

combined_aqi.to_csv("../../datascience/data/processed/combined_aqi_training.csv", index=False)

(21841, 12)
location_name    0.000000
date             0.000000
CO               0.114418
NO2              0.109656
O3               0.155625
PM10             0.147383
PM2.5            0.131358
SO2              0.153885
lat              0.000000
lon              0.000000
month            0.000000
day_of_week      0.000000
dtype: float64
Combined AQI training rows: 50350
Unique locations: 299


In [7]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import pickle

FEATURES = ["lat", "lon", "PM2.5", "PM10", "NO2", "month", "day_of_week"]
TARGET = "AQI"

df = combined_aqi.dropna(subset=FEATURES + [TARGET])
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(n_estimators=400, max_depth=7, learning_rate=0.05, subsample=0.8)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(f"Scaled AQI model — R²: {r2_score(y_test, preds):.3f}, MAE: {mean_absolute_error(y_test, preds):.2f}")

pickle.dump(model, open("models/aqi_xgboost_scaled.pkl", "wb"))

KeyError: ['AQI']

In [ ]:
print(combined_aqi.columns.tolist())
print(kaggle_df.columns.tolist())

['city', 'lat', 'lon', 'date', 'temp_max', 'temp_min', 'precipitation', 'location_name', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2', 'month', 'day_of_week']
['city', 'lat', 'lon', 'date', 'temp_max', 'temp_min', 'precipitation']


In [ ]:
kaggle_df["date"] = pd.to_datetime(kaggle_df["date"])
print(kaggle_df["date"].min(), kaggle_df["date"].max())
print(live_pivoted["date"].min(), live_pivoted["date"].max())

2023-01-01 00:00:00 2024-12-31 00:00:00
2026-05-16 00:00:00 2026-08-13 00:00:00


In [ ]:
from sklearn.neighbors import BallTree

cities = kaggle_df[["city", "lat", "lon"]].drop_duplicates("city").reset_index(drop=True)
tree = BallTree(np.radians(cities[["lat", "lon"]].values), metric="haversine")
dist, idx = tree.query(np.radians(live_pivoted[["lat", "lon"]].values), k=1)

live_pivoted["city"] = cities["city"].values[idx[:, 0]]
live_pivoted["dist_km"] = dist[:, 0] * 6371

# drop stations with no weather city nearby (e.g., Kathmandu)
live_pivoted = live_pivoted[live_pivoted["dist_km"] < 50]

weather = kaggle_df.drop(columns=["lat", "lon"])
combined_aqi = live_pivoted.merge(weather, on=["city", "date"], how="inner")

print(len(combined_aqi))
combined_aqi.to_csv("../../datascience/data/processed/combined_aqi_training.csv", index=False)

0


In [ ]:
import requests, time

cities = kaggle_df[["city", "lat", "lon"]].drop_duplicates("city")
rows = []

for _, c in cities.iterrows():
    r = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": c.lat, "longitude": c.lon,
            "start_date": "2026-05-16", "end_date": "2026-08-13",
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
            "timezone": "Asia/Kolkata",
        },
        timeout=30,
    )
    r.raise_for_status()
    d = r.json()["daily"]
    rows.append(pd.DataFrame({
        "city": c.city, "lat": c.lat, "lon": c.lon,
        "date": pd.to_datetime(d["time"]),
        "temp_max": d["temperature_2m_max"],
        "temp_min": d["temperature_2m_min"],
        "precipitation": d["precipitation_sum"],
    }))
    time.sleep(0.5)   # stay under the free rate limit

weather_2026 = pd.concat(rows, ignore_index=True)
weather_2026.to_csv("../../datascience/data/raw/weather_2026.csv", index=False)
print(weather_2026.shape)

(3510, 7)


In [ ]:
weather = weather_2026.drop(columns=["lat", "lon"])
combined_aqi = live_pivoted.merge(weather, on=["city", "date"], how="inner")
print(len(combined_aqi))

525


In [ ]:
print(len(combined_aqi), combined_aqi["location_name"].nunique())
print(combined_aqi[["PM2.5", "PM10", "NO2", "AQI", "temp_max", "temp_min", "precipitation"]].notna().sum())

525 6
PM2.5            427
PM10             427
NO2              434
AQI              427
temp_max         525
temp_min         525
precipitation    525
dtype: int64


In [ ]:
import numpy as np

# (conc_low, conc_high, aqi_low, aqi_high)
PM25_BP = [(0,30,0,50),(31,60,51,100),(61,90,101,200),(91,120,201,300),(121,250,301,400),(251,500,401,500)]
PM10_BP = [(0,50,0,50),(51,100,51,100),(101,250,101,200),(251,350,201,300),(351,430,301,400),(431,600,401,500)]

def sub_index(x, bp):
    if pd.isna(x):
        return np.nan
    for cl, ch, il, ih in bp:
        if x <= ch:
            return il + (x - cl) * (ih - il) / (ch - cl)
    return 500

combined_aqi["AQI"] = np.fmax(
    combined_aqi["PM2.5"].apply(lambda v: sub_index(v, PM25_BP)),
    combined_aqi["PM10"].apply(lambda v: sub_index(v, PM10_BP)),
)

print(combined_aqi["AQI"].describe())
print(combined_aqi[["PM2.5", "PM10", "NO2", "AQI", "temp_max", "temp_min", "precipitation"]].notna().sum())

count    427.000000
mean      88.827626
std       86.010754
min        0.120833
25%       38.800000
50%       63.600000
75%      110.966443
max      500.000000
Name: AQI, dtype: float64
PM2.5            427
PM10             427
NO2              434
AQI              427
temp_max         525
temp_min         525
precipitation    525
dtype: int64


In [ ]:
df = combined_aqi.sort_values(["location_name", "date"]).copy()
df["AQI_next"] = df.groupby("location_name")["AQI"].shift(-1)

# optional but useful: yesterday's AQI as a feature
df["AQI_prev"] = df.groupby("location_name")["AQI"].shift(1)

FEATURES = ["lat", "lon", "PM2.5", "PM10", "NO2", "AQI",
            "AQI_prev", "temp_max", "temp_min", "precipitation",
            "month", "day_of_week"]
TARGET = "AQI_next"

df = df.dropna(subset=FEATURES + [TARGET])

cutoff = df["date"].quantile(0.8)
train, test = df[df["date"] <= cutoff], df[df["date"] > cutoff]

model.fit(train[FEATURES], train[TARGET])
preds = model.predict(test[FEATURES])
print(r2_score(test[TARGET], preds), mean_absolute_error(test[TARGET], preds))

0.41873977125270456 17.05732342047157


In [ ]:
print("Persistence MAE:", mean_absolute_error(test[TARGET], test["AQI"]))
print("Persistence R2:", r2_score(test[TARGET], test["AQI"]))
print("Mean-AQI MAE:", mean_absolute_error(test[TARGET], [train[TARGET].mean()] * len(test)))

Persistence MAE: 11.309048273631438
Persistence R2: 0.7938025481171815
Mean-AQI MAE: 49.06518652721336


In [ ]:
df = combined_aqi.sort_values(["location_name", "date"]).copy()
g = df.groupby("location_name")

df["AQI_next"] = g["AQI"].shift(-1)
df["AQI_prev"] = g["AQI"].shift(1)
df.loc[(g["date"].shift(-1) - df["date"]).dt.days != 1, "AQI_next"] = np.nan
df.loc[(df["date"] - g["date"].shift(1)).dt.days != 1, "AQI_prev"] = np.nan

FEATURES = ["PM2.5", "PM10", "NO2", "AQI", "AQI_prev",
            "temp_max", "temp_min", "precipitation", "month", "day_of_week"]
TARGET = "AQI_next"

df = df.dropna(subset=FEATURES + [TARGET])
cutoff = df["date"].quantile(0.8)
train, test = df[df["date"] <= cutoff], df[df["date"] > cutoff]

model = xgb.XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8, min_child_weight=10)

model.fit(train[FEATURES], train[TARGET] - train["AQI"])      # learn the change
preds = test["AQI"] + model.predict(test[FEATURES])

print("Model MAE:", mean_absolute_error(test[TARGET], preds))
print("Model R2:", r2_score(test[TARGET], preds))
print("Persistence MAE:", mean_absolute_error(test[TARGET], test["AQI"]))

Model MAE: 16.756351037542913
Model R2: 0.5040275930167711
Persistence MAE: 11.495582363435563


In [ ]:
d_train = train[TARGET] - train["AQI"]
d_test  = test[TARGET]  - test["AQI"]

print(d_train.describe(), d_test.describe(), sep="\n\n")
print("Zero-change MAE:", d_test.abs().mean())
print("Train-median-change MAE:", (d_test - d_train.median()).abs().mean())
print("Stations train/test:", train["location_name"].nunique(), test["location_name"].nunique())
print(train[["AQI"]].describe().T, test[["AQI"]].describe().T, sep="\n")

count    315.000000
mean       2.257077
std       43.039767
min     -312.380555
25%      -11.707988
50%        1.328859
75%       12.650000
max      273.055814
dtype: float64

count    79.000000
mean      1.243769
std      14.742720
min     -25.716667
25%      -8.000000
50%       0.000000
75%      11.266667
max      48.340000
dtype: float64
Zero-change MAE: 11.495582363435563
Train-median-change MAE: 11.569504148613825
Stations train/test: 5 5
     count      mean        std       min   25%   50%         75%    max
AQI  315.0  98.80082  90.053064  0.120833  48.5  73.9  118.939597  500.0
     count       mean        std   min   25%   50%   75%         max
AQI   79.0  52.707456  32.780747  1.66  27.5  41.8  79.2  118.275168


In [ ]:
print("stations raw:", live_df["location_name"].nunique())
print("stations pivoted:", live_pivoted["location_name"].nunique())
print("stations after 50km filter:", live_pivoted[live_pivoted["dist_km"] < 50]["location_name"].nunique())
print("rows merged:", len(combined_aqi), "with AQI:", combined_aqi["AQI"].notna().sum())
print("rows with all of PM2.5, PM10, NO2:", combined_aqi[["PM2.5","PM10","NO2"]].notna().all(axis=1).sum())

stations raw: 260
stations pivoted: 6
stations after 50km filter: 6
rows merged: 525 with AQI: 427
rows with all of PM2.5, PM10, NO2: 427


In [ ]:
import requests, time, numpy as np, pandas as pd

# 1. Pivot (no city filter)
live_df["date"] = pd.to_datetime(live_df["date"].astype(str).str[:10])
loc = live_df.groupby("location_name")[["lat", "lon"]].first().reset_index()

live_pivoted = live_df.pivot_table(index=["location_name", "date"], columns="parameter",
                                   values="value", aggfunc="mean").reset_index()
live_pivoted.columns.name = None
live_pivoted = live_pivoted.rename(columns={"pm25": "PM2.5", "pm10": "PM10", "no2": "NO2",
                                            "so2": "SO2", "co": "CO", "o3": "O3"})
live_pivoted = live_pivoted.merge(loc, on="location_name")
live_pivoted["month"] = live_pivoted["date"].dt.month
live_pivoted["day_of_week"] = live_pivoted["date"].dt.dayofweek
live_pivoted["AQI"] = np.fmax(live_pivoted["PM2.5"].apply(lambda v: sub_index(v, PM25_BP)),
                              live_pivoted["PM10"].apply(lambda v: sub_index(v, PM10_BP)))
print(live_pivoted["location_name"].nunique(), len(live_pivoted))

# 2. Weather at each station (batched, 50 stations per request)
start, end = live_pivoted["date"].min().date(), live_pivoted["date"].max().date()
rows = []
for i in range(0, len(loc), 50):
    chunk = loc.iloc[i:i+50]
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": ",".join(chunk.lat.round(4).astype(str)),
        "longitude": ",".join(chunk.lon.round(4).astype(str)),
        "start_date": str(start), "end_date": str(end),
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
        "timezone": "auto"}, timeout=60)
    r.raise_for_status()
    res = r.json()
    res = res if isinstance(res, list) else [res]
    for name, d in zip(chunk.location_name, res):
        rows.append(pd.DataFrame({"location_name": name,
                                  "date": pd.to_datetime(d["daily"]["time"]),
                                  "temp_max": d["daily"]["temperature_2m_max"],
                                  "temp_min": d["daily"]["temperature_2m_min"],
                                  "precipitation": d["daily"]["precipitation_sum"],
                                  "wind_max": d["daily"]["wind_speed_10m_max"]}))
    time.sleep(2)

weather_st = pd.concat(rows, ignore_index=True)
combined_aqi = live_pivoted.merge(weather_st, on=["location_name", "date"], how="left")
print(len(combined_aqi), combined_aqi["AQI"].notna().sum(), combined_aqi["temp_max"].notna().sum())

260 21841
21841 19640 21841


In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

df = combined_aqi.sort_values(["location_name", "date"]).copy()
g = df.groupby("location_name")

df["AQI_next"] = g["AQI"].shift(-1)
df["AQI_prev"] = g["AQI"].shift(1)
df.loc[(g["date"].shift(-1) - df["date"]).dt.days != 1, "AQI_next"] = np.nan
df.loc[(df["date"] - g["date"].shift(1)).dt.days != 1, "AQI_prev"] = np.nan

FEATURES = ["PM2.5", "PM10", "AQI", "AQI_prev",
            "temp_max", "temp_min", "precipitation", "month", "day_of_week"]
TARGET = "AQI_next"

df = df.dropna(subset=FEATURES + [TARGET])
print("usable rows:", len(df), "stations:", df["location_name"].nunique())

cutoff = df["date"].quantile(0.8)
train, test = df[df["date"] <= cutoff], df[df["date"] > cutoff]
print("train/test:", len(train), len(test), "| test stations:", test["location_name"].nunique())

d_train = train[TARGET] - train["AQI"]
lo, hi = d_train.quantile([0.02, 0.98])

model = xgb.XGBRegressor(objective="reg:absoluteerror", n_estimators=300, max_depth=4,
                         learning_rate=0.05, min_child_weight=20,
                         subsample=0.8, colsample_bytree=0.8)
model.fit(train[FEATURES], d_train.clip(lo, hi))
raw = model.predict(test[FEATURES])

print("Persistence MAE:", mean_absolute_error(test[TARGET], test["AQI"]))
print("Persistence R2: ", r2_score(test[TARGET], test["AQI"]))
for w in [0, 0.5, 1.0]:
    p = test["AQI"] + w * raw
    print(f"w={w}  MAE={mean_absolute_error(test[TARGET], p):.2f}  R2={r2_score(test[TARGET], p):.3f}")

usable rows: 16742 stations: 215
train/test: 13446 3296 | test stations: 211
Persistence MAE: 16.13139272474192
Persistence R2:  0.36355345183578436
w=0  MAE=16.13  R2=0.364
w=0.5  MAE=15.69  R2=0.405
w=1.0  MAE=15.62  R2=0.422


In [ ]:
w = weather_st.sort_values(["location_name", "date"]).copy()
for c in ["temp_max", "precipitation", "wind_max"]:
    w[c + "_next"] = w.groupby("location_name")[c].shift(-1)
combined_aqi = combined_aqi.merge(
    w[["location_name", "date", "temp_max_next", "precipitation_next", "wind_max_next"]],
    on=["location_name", "date"], how="left")

In [ ]:
import pickle, os, numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import mean_absolute_error

# ---------- build features ----------
df = combined_aqi.sort_values(["location_name", "date"]).copy()
g = df.groupby("location_name")
df["AQI_next"] = g["AQI"].shift(-1)
df["AQI_prev"] = g["AQI"].shift(1)
df.loc[(g["date"].shift(-1) - df["date"]).dt.days != 1, "AQI_next"] = np.nan
df.loc[(df["date"] - g["date"].shift(1)).dt.days != 1, "AQI_prev"] = np.nan
df["AQI_diff"] = df["AQI"] - df["AQI_prev"]
df["AQI_roll3"] = df.groupby("location_name")["AQI"].transform(
    lambda s: s.rolling(3, min_periods=2).mean())

BASE = ["PM2.5", "PM10", "AQI", "AQI_prev", "AQI_diff", "AQI_roll3", "month", "day_of_week"]
WX_TODAY = ["temp_max", "temp_min", "precipitation"]
WX_NEXT = ["temp_max_next", "precipitation_next", "wind_max_next"]
FEATURE_SETS = {"aqi_only": BASE,
                "with_weather": BASE + WX_TODAY + WX_NEXT}
TARGET = "AQI_next"

def make_model():
    return xgb.XGBRegressor(objective="reg:absoluteerror", n_estimators=300, max_depth=4,
                            learning_rate=0.05, min_child_weight=20,
                            subsample=0.8, colsample_bytree=0.8, random_state=42)

def fit_eval(feats, tr, te):
    d = tr[TARGET] - tr["AQI"]
    lo, hi = d.quantile([0.02, 0.98])
    m = make_model().fit(tr[feats], d.clip(lo, hi))
    pred = (te["AQI"] + m.predict(te[feats])).clip(0, 500)
    return mean_absolute_error(te[TARGET], pred), (lo, hi)

# ---------- pick feature set on validation, report on test ----------
all_feats = sorted(set(sum(FEATURE_SETS.values(), [])))
data = df.dropna(subset=all_feats + [TARGET])
c1, c2 = data["date"].quantile([0.6, 0.8])
train, val, test = (data[data["date"] <= c1],
                    data[(data["date"] > c1) & (data["date"] <= c2)],
                    data[data["date"] > c2])

scores = {k: fit_eval(f, train, val)[0] for k, f in FEATURE_SETS.items()}
best = min(scores, key=scores.get)
feats = FEATURE_SETS[best]
print("validation MAE:", {k: round(v, 2) for k, v in scores.items()}, "-> chosen:", best)

trval = data[data["date"] <= c2]
test_mae, _ = fit_eval(feats, trval, test)
print(f"TEST  model MAE: {test_mae:.2f} | persistence MAE: "
      f"{mean_absolute_error(test[TARGET], test['AQI']):.2f}")

# ---------- refit on everything and save ----------
final = data
d = final[TARGET] - final["AQI"]
lo, hi = d.quantile([0.02, 0.98])
model = make_model().fit(final[feats], d.clip(lo, hi))

os.makedirs("models", exist_ok=True)
bundle = {
    "model": model,
    "features": feats,
    "feature_set": best,
    "predicts": "delta",   # next-day AQI = today's AQI + model output
    "test_mae": test_mae,
    "trained_through": str(final["date"].max().date()),
}
pickle.dump(bundle, open("models/aqi_forecast_final.pkl", "wb"))
model.save_model("models/aqi_forecast_final.json")   # version-safe backup
print("saved models/aqi_forecast_final.pkl with features:", feats)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:wind_max_next: object

In [ ]:
print([c for c in ["wind_max","temp_max_next","precipitation_next","wind_max_next"] if c not in combined_aqi.columns])

['wind_max']
